<a href="https://colab.research.google.com/github/kawastony/Quantum_Gravity/blob/main/Geometry_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize, minimize_scalar
from scipy.stats import spearmanr

# ── Constants ────────────────────────────────────────────
G_kpc        = 4.30091e-6   # kpc (km/s)^2 / M_sun
Reff_to_h    = 1.678
RHI_to_h_gas = 3.5
hz_base      = 2.0
alpha        = 0.25
mu           = 0.0824
SIGMA_MOND   = 138e6        # M_sun / kpc^2

# ── SPARC column spec (24 columns, exact order) ──────────
SPARC_COLS = [
    'Galaxy', 'T', 'D', 'e_D', 'f_D',
    'Inc', 'e_Inc', 'L36', 'e_L36', 'Reff',
    'SBeff', 'Rdisk', 'SBdisk', 'MHI', 'RHI',
    'Vflat', 'e_Vflat', 'Q', 'Mbar', 'fgas',
    'Meff', 'eta', 'Meff_dyn', 'rp_dyn'
]

def load_sparc(file_path='/content/SPARC_Lelli2016c.mrt.txt'):
    """
    Robust SPARC loader.
    Skips all header/comment lines (anything that does not
    start with a letter — galaxy names always do).
    Returns cleaned DataFrame with correct column mapping.
    """
    data_lines = []
    with open(file_path, 'r') as f:
        for line in f:
            stripped = line.strip()
            if not stripped:
                continue
            # Data rows start with a galaxy name (letter)
            if stripped[0].isalpha():
                data_lines.append(stripped)

    if not data_lines:
        raise ValueError("No data rows found — check file path.")

    # Parse into DataFrame
    records = []
    for line in data_lines:
        parts = line.split()
        if len(parts) < len(SPARC_COLS):
            # Pad with NaN if short (shouldn't happen)
            parts += [np.nan] * (len(SPARC_COLS) - len(parts))
        records.append(parts[:len(SPARC_COLS)])

    df = pd.DataFrame(records, columns=SPARC_COLS)

    # Convert numeric columns
    for col in SPARC_COLS[1:]:   # everything except Galaxy
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Quality cuts
    df = df.dropna(subset=['Vflat', 'L36', 'MHI'])
    df = df[(df['Vflat'] > 0) & (df['L36'] > 0) & (df['MHI'] > 0)]
    df = df.reset_index(drop=True)

    print(f"Loaded {len(df)} galaxies.")
    print(f"  L36   : {df['L36'].min():.3f} – "
          f"{df['L36'].max():.3f}  "
          f"(mean {df['L36'].mean():.3f})  [1e9 L_sun]")
    print(f"  Vflat : {df['Vflat'].min():.1f} – "
          f"{df['Vflat'].max():.1f}  "
          f"(mean {df['Vflat'].mean():.1f})  [km/s]")
    print(f"  Reff  : {df['Reff'].min():.3f} – "
          f"{df['Reff'].max():.3f}  "
          f"(mean {df['Reff'].mean():.3f})  [kpc]")
    print(f"  MHI   : {df['MHI'].min():.3f} – "
          f"{df['MHI'].max():.3f}  [1e9 M_sun]")
    return df

# ── Galaxy-level computation ──────────────────────────────

def get_upsilon(T):
    """M/L ratio at 3.6 µm. Early types get 0.7, late 0.5."""
    if pd.isna(T):
        return 0.5
    return 0.7 if T <= 2 else 0.5

def enclosed_exp(r, h):
    """Fraction of exponential disk mass within radius r."""
    if pd.isna(h) or h <= 0 or r <= 0:
        return 1.0
    u = r / h
    return 1.0 - np.exp(-u) * (1.0 + u)

def galaxy_cone(row):
    """
    Compute M_cone and Sigma_bar(r_p) for one galaxy.
    Returns (M_cone, Sigma_bar, r_p, valid).
    """
    T      = row['T']   if pd.notna(row['T'])   else np.nan
    L36    = row['L36']
    MHI    = row['MHI']
    Reff   = row['Reff']  if pd.notna(row['Reff'])  else np.nan
    Rdisk  = row['Rdisk'] if pd.notna(row['Rdisk']) else np.nan
    RHI    = row['RHI']   if pd.notna(row['RHI'])   else np.nan

    ups    = get_upsilon(T)
    M_star = ups  * L36  * 1e9   # M_sun
    M_gas  = 1.33 * MHI  * 1e9  # M_sun (HI + He)

    # Stellar scale
    if pd.notna(Reff) and Reff > 0:
        R_eff = Reff
    elif pd.notna(Rdisk) and Rdisk > 0:
        R_eff = Reff_to_h * Rdisk
    else:
        return None, None, None, False   # cannot proceed

    h_star = R_eff / Reff_to_h

    # Gas scale
    if pd.notna(RHI) and RHI > 0:
        h_gas = RHI / RHI_to_h_gas
    else:
        h_gas = 2.5 * h_star   # fallback: h_gas ~ 2.5 h_star

    # Cone aperture
    r_p    = 3.0 * R_eff

    # Enclosed masses
    M_se   = M_star * enclosed_exp(r_p, h_star)
    M_ge   = M_gas  * enclosed_exp(r_p, h_gas)
    M_cone = max(M_se + (1.0/hz_base) * M_ge, 1e6)

    # Surface density at cone aperture
    Sigma_bar = M_cone / (np.pi * r_p**2)

    return M_cone, Sigma_bar, r_p, True

# ── Metric computation ────────────────────────────────────

def compute_metrics(df, beta_chi, logL,
                    Sigma_crit=SIGMA_MOND,
                    full=False):
    """
    Compute MAE, bias, and Spearman correlations
    for the chiral running-Lambda model.

    If full=True, also return per-galaxy diagnostics.
    """
    L0     = 10**logL
    resids = []
    vflat  = []
    sigma  = []
    fgas   = []
    reff   = []
    leff   = []

    for _, row in df.iterrows():
        Vobs = row['Vflat']
        if Vobs <= 0:
            continue

        M_cone, Sigma_bar, r_p, ok = galaxy_cone(row)
        if not ok:
            continue

        # Chiral suppression
        xi       = max(1.0 + beta_chi * Sigma_bar / Sigma_crit,
                       1e-6)
        Leff     = L0 / np.sqrt(xi)
        Vp       = (Leff**2 * G_kpc * M_cone * mu)**alpha

        if Vp <= 0:
            continue

        resid = np.log10(Vobs) - np.log10(Vp)
        resids.append(resid)
        vflat.append(Vobs)
        sigma.append(Sigma_bar)

        T   = row['T']   if pd.notna(row['T'])   else np.nan
        ups = get_upsilon(T)
        Ms  = ups  * row['L36'] * 1e9
        Mg  = 1.33 * row['MHI'] * 1e9
        fgas.append(Mg / (Ms + Mg))

        Reff_val = row['Reff'] if pd.notna(row['Reff']) else np.nan
        reff.append(Reff_val)
        leff.append(Leff)

    if len(resids) < 5:
        return {'MAE': 999, 'Bias': 999,
                'rho_V': 0, 'rho_S': 0,
                'rho_g': 0, 'rho_r': 0,
                'N': 0}

    arr  = np.array(resids)
    rV,_ = spearmanr(vflat, arr)
    rS,_ = spearmanr(sigma,  arr)
    rG,_ = spearmanr(fgas,   arr)
    re   = [(v,r) for v,r in zip(reff,arr)
            if np.isfinite(v)]
    rR   = spearmanr([x[0] for x in re],
                     [x[1] for x in re])[0] if re else 0

    out = {
        'MAE':   np.mean(np.abs(arr)),
        'Bias':  np.mean(arr),
        'rho_V': rV,
        'rho_S': rS,
        'rho_g': rG,
        'rho_r': rR,
        'N':     len(arr),
    }

    if full:
        out['resids'] = arr
        out['vflat']  = np.array(vflat)
        out['sigma']  = np.array(sigma)
        out['fgas']   = np.array(fgas)
        out['reff']   = np.array(reff)
        out['leff']   = np.array(leff)

    return out

# ── Load and validate ─────────────────────────────────────

dev = load_sparc()

# Sanity check on a known galaxy before optimising
print("\n--- Sanity check: UGC06614 ---")
if 'UGC06614' in dev['Galaxy'].values:
    row = dev[dev['Galaxy']=='UGC06614'].iloc[0]
    Mc, Sb, rp, ok = galaxy_cone(row)
    print(f"  M_cone    = {Mc:.3e} M_sun")
    print(f"  Sigma_bar = {Sb:.3e} M_sun/kpc^2")
    print(f"  r_p       = {rp:.3f} kpc")
    print(f"  Vflat     = {row['Vflat']:.1f} km/s")
    print(f"  L36       = {row['L36']:.3f} [1e9 L_sun]")
else:
    # Check first galaxy
    row = dev.iloc[0]
    print(f"  Galaxy    = {row['Galaxy']}")
    Mc, Sb, rp, ok = galaxy_cone(row)
    print(f"  M_cone    = {Mc:.3e} M_sun")
    print(f"  Sigma_bar = {Sb:.3e} M_sun/kpc^2")
    print(f"  r_p       = {rp:.3f} kpc")
    print(f"  Vflat     = {row['Vflat']:.1f} km/s")
    print(f"  L36       = {row['L36']:.3f} [1e9 L_sun]")

# ── Baseline: beta=0, scan Lambda* ───────────────────────

print("\n--- Baseline Lambda* scan (beta=0) ---")
logL_grid = np.linspace(1.5, 3.5, 200)
maes      = [compute_metrics(dev, 0.0, lL)['MAE']
             for lL in logL_grid]
logL_base = logL_grid[np.argmin(maes)]
m_base    = compute_metrics(dev, 0.0, logL_base)
print(f"  Lambda*  = {10**logL_base:.2f}")
print(f"  MAE      = {m_base['MAE']:.5f} dex")
print(f"  Bias     = {m_base['Bias']:+.5f} dex")
print(f"  rho_V    = {m_base['rho_V']:+.4f}")
print(f"  rho_Reff = {m_base['rho_r']:+.4f}")
print(f"  N        = {m_base['N']}")

# ── Beta scan at fixed Sigma_crit = MOND ─────────────────

print(f"\n{'='*70}")
print(f"CHIRAL RUNNING SCAN")
print(f"Sigma_crit = {SIGMA_MOND:.2e} M_sun/kpc^2  (MOND reference)")
print(f"{'='*70}")
print(f"\n{'beta':>8} {'Lambda*':>9} {'MAE':>9} "
      f"{'Bias':>8} {'rhoV':>7} {'pV':>10} "
      f"{'rhoSig':>8} {'rhoGas':>8} {'rhoReff':>8}")
print("-" * 80)

best_mae  = 999
best_beta = 0.0
best_logL = logL_base

for beta in [0.00, 0.01, 0.05, 0.10, 0.20,
             0.50, 1.00, 2.00, 5.00,
             10.0, 20.0, 50.0, 100.0]:

    # Optimise Lambda* for this beta
    res = minimize_scalar(
        lambda lL: compute_metrics(
            dev, beta, lL, SIGMA_MOND)['MAE'],
        bounds=(1.5, 3.5),
        method='bounded')

    lL_opt = res.x
    m      = compute_metrics(dev, beta, lL_opt, SIGMA_MOND)
    rV, pV = spearmanr(
        *[x for x in [
            compute_metrics(dev, beta, lL_opt,
                            SIGMA_MOND, full=True).get(
                                'vflat', []),
            compute_metrics(dev, beta, lL_opt,
                            SIGMA_MOND, full=True).get(
                                'resids', [])]
        ]
    ) if False else (m['rho_V'],
                     spearmanr(range(m['N']),
                                range(m['N']))[1])

    # Get p-value properly
    mf   = compute_metrics(dev, beta, lL_opt,
                           SIGMA_MOND, full=True)
    rV2, pV2 = spearmanr(mf['vflat'], mf['resids'])

    if m['MAE'] < best_mae:
        best_mae  = m['MAE']
        best_beta = beta
        best_logL = lL_opt

    sig = ("***" if pV2 < 0.001 else
           "**"  if pV2 < 0.01  else
           "*"   if pV2 < 0.05  else
           "."   if pV2 < 0.10  else " ")
    mark = "  ← baseline" if beta == 0.0 else ""

    print(f"{beta:>8.2f} {10**lL_opt:>9.2f} "
          f"{m['MAE']:>9.5f} {m['Bias']:>+8.5f} "
          f"{rV2:>+7.4f} {pV2:>10.3e}{sig} "
          f"{m['rho_S']:>+8.4f} "
          f"{m['rho_g']:>+8.4f} "
          f"{m['rho_r']:>+8.4f}{mark}")

# ── 2D optimisation (beta and Lambda* free) ──────────────

print("\n--- 2D optimisation (beta, Lambda* free) ---")

def obj2(params):
    beta = max(params[0], 0.0)
    logL = params[1]
    if logL < 1.5 or logL > 3.5:
        return 999.0
    return compute_metrics(dev, beta, logL,
                           SIGMA_MOND)['MAE']

res2 = minimize(obj2,
                x0=[best_beta, best_logL],
                method='Nelder-Mead',
                options={'xatol':1e-6,
                         'fatol':1e-7,
                         'maxiter':10000})

beta2 = max(res2.x[0], 0.0)
logL2 = res2.x[1]
m2    = compute_metrics(dev, beta2, logL2,
                        SIGMA_MOND, full=True)
rV2, pV2 = spearmanr(m2['vflat'], m2['resids'])
rS2, pS2 = spearmanr(m2['sigma'],  m2['resids'])

print(f"\n  beta_chi  = {beta2:.5f}")
print(f"  Lambda*   = {10**logL2:.4f}")
print(f"  MAE       = {m2['MAE']:.5f} dex")
print(f"  Bias      = {m2['Bias']:+.5f} dex")
print(f"  rho_V     = {rV2:+.4f}  p={pV2:.3e}")
print(f"  rho_Sigma = {rS2:+.4f}  p={pS2:.3e}")
print(f"  rho_gas   = {m2['rho_g']:+.4f}")
print(f"  rho_Reff  = {m2['rho_r']:+.4f}")
print(f"  N         = {m2['N']}")

# ── 3D optimisation (Sigma_crit free too) ────────────────

print("\n--- 3D optimisation (Sigma_crit free) ---")

def obj3(params):
    beta  = max(params[0], 0.0)
    logL  = params[1]
    logSc = params[2]
    if logL < 1.5 or logL > 3.5:
        return 999.0
    if logSc < 4.0 or logSc > 11.0:
        return 999.0
    Sc = 10**logSc
    return compute_metrics(dev, beta, logL, Sc)['MAE']

res3 = minimize(obj3,
                x0=[beta2, logL2,
                    np.log10(SIGMA_MOND)],
                method='Nelder-Mead',
                options={'xatol':1e-6,
                         'fatol':1e-7,
                         'maxiter':20000})

beta3 = max(res3.x[0], 0.0)
logL3 = res3.x[1]
Sc3   = 10**res3.x[2]
m3    = compute_metrics(dev, beta3, logL3,
                        Sc3, full=True)
rV3, pV3 = spearmanr(m3['vflat'], m3['resids'])
rS3, pS3 = spearmanr(m3['sigma'],  m3['resids'])

print(f"\n  beta_chi    = {beta3:.5f}")
print(f"  Lambda*     = {10**logL3:.4f}")
print(f"  Sigma_crit  = {Sc3:.4e} M_sun/kpc^2")
print(f"  MOND ratio  = {Sc3/SIGMA_MOND:.4f} × MOND ref")
print(f"  MAE         = {m3['MAE']:.5f} dex")
print(f"  Bias        = {m3['Bias']:+.5f} dex")
print(f"  rho_V       = {rV3:+.4f}  p={pV3:.3e}")
print(f"  rho_Sigma   = {rS3:+.4f}  p={pS3:.3e}")
print(f"  rho_gas     = {m3['rho_g']:+.4f}")
print(f"  rho_Reff    = {m3['rho_r']:+.4f}")
print(f"  N           = {m3['N']}")

# ── Full residual audit at 3D optimum ────────────────────

print("\n--- Full residual correlations at 3D optimum ---")
df_diag = pd.DataFrame({
    'resid':     m3['resids'],
    'Vflat':     m3['vflat'],
    'Sigma_bar': m3['sigma'],
    'fgas':      m3['fgas'],
    'R_eff':     m3['reff'],
    'Lambda_eff':m3['leff'],
})

# Attach T and Xi from dev (aligned by position)
valid_idx = []
for i, row in dev.iterrows():
    Mc, Sb, rp, ok = galaxy_cone(row)
    if ok and row['Vflat'] > 0:
        valid_idx.append(i)

if len(valid_idx) == len(df_diag):
    df_diag['T']  = dev.loc[valid_idx, 'T'].values
    df_diag['RHI'] = dev.loc[valid_idx, 'RHI'].values
    df_diag['Xi']  = (df_diag['RHI'] /
                      df_diag['R_eff'].replace(0, np.nan))

print(f"\n  N={len(df_diag)}"
      f"  MAE={df_diag['resid'].abs().mean():.5f}"
      f"  Bias={df_diag['resid'].mean():+.5f}")

for col, name in [
    ('Vflat',     'Vflat'),
    ('R_eff',     'R_eff'),
    ('fgas',      'fgas'),
    ('Sigma_bar', 'Sigma_bar'),
    ('Lambda_eff','Lambda_eff'),
    ('T',         'T'),
    ('Xi',        'R_HI/R_eff'),
]:
    if col not in df_diag.columns:
        continue
    sub = df_diag[['resid',col]].dropna()
    if len(sub) < 5:
        continue
    rho, p = spearmanr(sub[col], sub['resid'])
    sig = ("***" if p<0.001 else "**" if p<0.01
           else "*" if p<0.05 else "." if p<0.10 else "")
    print(f"  rho(resid, {name:<12}): "
          f"{rho:+.4f}  p={p:.3e}  {sig}")

# ── Summary ───────────────────────────────────────────────
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"  {'Model':<32} {'MAE':>8} "
      f"{'rhoV':>8} {'rhoReff':>9}")
print(f"  {'-'*59}")
print(f"  {'Baseline (beta=0)':<32} "
      f"{m_base['MAE']:>8.5f} "
      f"{m_base['rho_V']:>+8.4f} "
      f"{m_base['rho_r']:>+9.4f}")
print(f"  {'Chiral 2D (MOND Sigma_crit)':<32} "
      f"{m2['MAE']:>8.5f} "
      f"{rV2:>+8.4f} "
      f"{m2['rho_r']:>+9.4f}")
print(f"  {'Chiral 3D (Sigma_crit free)':<32} "
      f"{m3['MAE']:>8.5f} "
      f"{rV3:>+8.4f} "
      f"{m3['rho_r']:>+9.4f}")

Loaded 135 galaxies.
  L36   : 0.012 – 489.955  (mean 63.406)  [1e9 L_sun]
  Vflat : 18.6 – 332.0  (mean 135.3)  [km/s]
  Reff  : 0.320 – 12.360  (mean 3.902)  [kpc]
  MHI   : 0.067 – 40.075  [1e9 M_sun]

--- Sanity check: UGC06614 ---
  M_cone    = 8.558e+10 M_sun
  Sigma_bar = 2.235e+08 M_sun/kpc^2
  r_p       = 11.040 kpc
  Vflat     = 199.8 km/s
  L36       = 124.350 [1e9 L_sun]

--- Baseline Lambda* scan (beta=0) ---
  Lambda*  = 278.43
  MAE      = 0.05678 dex
  Bias     = +0.00006 dex
  rho_V    = +0.0780
  rho_Reff = -0.2494
  N        = 135

CHIRAL RUNNING SCAN
Sigma_crit = 1.38e+08 M_sun/kpc^2  (MOND reference)

    beta   Lambda*       MAE     Bias    rhoV         pV   rhoSig   rhoGas  rhoReff
--------------------------------------------------------------------------------
    0.00    281.40   0.05673 -0.00224 +0.0780  3.687e-01   +0.0203  +0.0656  -0.2494  ← baseline
    0.01    282.04   0.05667 -0.00221 +0.0829  3.389e-01   +0.0275  +0.0605  -0.2504
    0.05    285.99   0.

In [9]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from numpy.linalg import lstsq

# ── Recompute residuals and attach all predictors ────────

rows = []
for _, row in dev.iterrows():
    M_cone, Sigma_bar, r_p, ok = galaxy_cone(row)
    if not ok or row['Vflat'] <= 0:
        continue

    T   = row['T']   if pd.notna(row['T'])   else np.nan
    ups = get_upsilon(T)
    Ms  = ups  * row['L36'] * 1e9
    Mg  = 1.33 * row['MHI'] * 1e9
    Reff = row['Reff'] if pd.notna(row['Reff']) else np.nan
    RHI  = row['RHI']  if pd.notna(row['RHI'])  else np.nan

    Leff = 10**logL_base          # baseline, beta=0
    Vp   = (Leff**2 * G_kpc * M_cone * mu)**alpha
    if Vp <= 0:
        continue

    resid = np.log10(row['Vflat']) - np.log10(Vp)

    rows.append({
        'Galaxy':     row['Galaxy'],
        'resid':      resid,
        'Vflat':      row['Vflat'],
        'L36':        row['L36'],
        'R_eff':      Reff,
        'logReff':    np.log10(Reff) if pd.notna(Reff) and Reff>0 else np.nan,
        'T':          T,
        'fgas':       Mg/(Ms+Mg),
        'logMstar':   np.log10(Ms),
        'logMgas':    np.log10(Mg),
        'logMbar':    np.log10(Ms+Mg),
        'Sigma_bar':  Sigma_bar,
        'logSigma':   np.log10(Sigma_bar),
        'Xi':         (RHI/Reff if pd.notna(RHI) and pd.notna(Reff)
                       and Reff>0 else np.nan),
        'r_p':        r_p,
        'fenc_star':  enclosed_exp(r_p, Reff/Reff_to_h),
        'fenc_gas':   enclosed_exp(r_p, RHI/RHI_to_h_gas
                                   if pd.notna(RHI) and RHI>0
                                   else 2.5*Reff/Reff_to_h),
    })

df = pd.DataFrame(rows)
print(f"N = {len(df)}")
print(f"MAE  = {df['resid'].abs().mean():.5f}")
print(f"Bias = {df['resid'].mean():+.5f}")

# ── 1. Marginal correlations ─────────────────────────────

print("\n--- Marginal Spearman correlations with residual ---")
for col in ['Vflat','R_eff','logReff','T','fgas',
            'logMstar','logMbar','Sigma_bar',
            'logSigma','Xi','fenc_star','fenc_gas']:
    sub = df[['resid',col]].dropna()
    if len(sub) < 10:
        continue
    rho, p = spearmanr(sub[col], sub['resid'])
    sig = ("***" if p<0.001 else "**" if p<0.01
           else "*" if p<0.05 else "." if p<0.10 else "")
    print(f"  rho(resid, {col:<12}): "
          f"{rho:+.4f}  p={p:.3e}  {sig}")

# ── 2. Partial: rho(resid, R_eff | logMbar) ─────────────

print("\n--- Partial correlations ---")

def partial_spearman(df_in, y, x, controls):
    cols   = [y, x] + controls
    sub    = df_in[cols].dropna()
    ranked = sub.rank()

    def resid_from_controls(target):
        Y = ranked[target].values
        C = ranked[controls].values
        if C.ndim == 1:
            C = C.reshape(-1,1)
        A = np.column_stack([C, np.ones(len(C))])
        coef,_,_,_ = lstsq(A, Y, rcond=None)
        return Y - A @ coef

    ry = resid_from_controls(y)
    rx = resid_from_controls(x)
    return spearmanr(rx, ry)

rho, p = partial_spearman(df, 'resid', 'R_eff', ['logMbar'])
print(f"  rho(resid, R_eff  | logMbar): {rho:+.4f}  p={p:.3e}")

# ── 4. Implied Upsilon from Vflat ────────────────────────

print("\n--- Implied M/L test ---")
Lopt = 10**logL_base
implied = []
for _, row in df.iterrows():
    T, ups = row['T'], get_upsilon(row['T'])
    Mg = 1.33 * dev[dev['Galaxy']==row['Galaxy']]['MHI'].values[0] * 1e9
    Reff = row['R_eff']
    h_star = Reff / Reff_to_h
    match = dev[dev['Galaxy']==row['Galaxy']]
    RHI = match['RHI'].values[0] if len(match)>0 else np.nan
    h_gas = (RHI/RHI_to_h_gas if pd.notna(RHI) and RHI>0 else 2.5*h_star)
    r_p = 3.0 * Reff
    Mse = enclosed_exp(r_p, h_star)
    Mge = Mg * enclosed_exp(r_p, h_gas)
    M_need = row['Vflat']**(1/alpha) / (Lopt**2 * G_kpc * mu)**(1/alpha)
    Ups_need = ((M_need - (1/hz_base)*Mge) / (row['L36']*1e9 * Mse))
    implied.append({'Galaxy': row['Galaxy'], 'R_eff': Reff, 'T': T, 'fgas': row['fgas'], 'Ups_model': ups, 'Ups_need': Ups_need})

di = pd.DataFrame(implied)
di = di[di['Ups_need'] > 0]
rho_ru, p_ru = spearmanr(di['R_eff'], di['Ups_need'])
print(f"  rho(Ups_need, R_eff): {rho_ru:+.4f}  p={p_ru:.3e}")

N = 135
MAE  = 0.05678
Bias = +0.00006

--- Marginal Spearman correlations with residual ---
  rho(resid, Vflat       ): +0.0780  p=3.687e-01  
  rho(resid, R_eff       ): -0.2494  p=3.528e-03  **
  rho(resid, logReff     ): -0.2494  p=3.528e-03  **
  rho(resid, T           ): +0.0570  p=5.114e-01  
  rho(resid, fgas        ): +0.0656  p=4.494e-01  
  rho(resid, logMstar    ): -0.1529  p=7.663e-02  .
  rho(resid, logMbar     ): -0.1756  p=4.165e-02  *
  rho(resid, Sigma_bar   ): +0.0203  p=8.153e-01  
  rho(resid, logSigma    ): +0.0203  p=8.153e-01  
  rho(resid, Xi          ): +0.1115  p=1.981e-01  
  rho(resid, fenc_star   ): +0.0500  p=5.648e-01  
  rho(resid, fenc_gas    ): -0.1249  p=1.490e-01  

--- Partial correlations ---
  rho(resid, R_eff  | logMbar): -0.1987  p=2.088e-02

--- Implied M/L test ---
  rho(Ups_need, R_eff): -0.4128  p=6.519e-07


In [10]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize, minimize_scalar
from scipy.stats import spearmanr

G_kpc        = 4.30091e-6
Reff_to_h    = 1.678
RHI_to_h_gas = 3.5
hz_base      = 2.0
alpha        = 0.25
mu           = 0.0824

def enclosed_exp(r, h):
    if pd.isna(h) or h <= 0 or r <= 0:
        return 1.0
    u = r / h
    return 1.0 - np.exp(-u) * (1.0 + u)

# ── Model A: constant Upsilon (current baseline) ─────────

def upsilon_const(row, ups_E=0.7, ups_L=0.5):
    T = row['T'] if pd.notna(row['T']) else np.nan
    return ups_E if (pd.notna(T) and T <= 2) else ups_L

# ── Model B: Upsilon from stellar surface density ────────
# log(Ups) = ups0 + gamma * log(Sigma_star)
# Sigma_star = L36 / (2 * pi * Reff^2)   [1e9 L_sun / kpc^2]

def upsilon_sigma(row, ups0, gamma):
    L36  = row['L36']
    Reff = row['Reff'] if pd.notna(row['Reff']) else np.nan
    if pd.isna(Reff) or Reff <= 0 or L36 <= 0:
        return upsilon_const(row)  # fallback
    Sigma_star = L36 / (2.0 * np.pi * Reff**2)
    logS       = np.log10(Sigma_star)
    ups        = 10**(ups0 + gamma * logS)
    # Physical bounds: 0.1 < Ups < 3.0
    return float(np.clip(ups, 0.1, 3.0))

# ── Model C: Upsilon from SBeff (if column exists) ───────
# log(Ups) = a + b * SBeff
# SBeff in SPARC is mag/arcsec^2

def upsilon_SBeff(row, a, b):
    SBeff = row.get('SBeff', np.nan)
    if pd.isna(SBeff):
        return upsilon_const(row)
    ups = 10**(a + b * SBeff)
    return float(np.clip(ups, 0.1, 3.0))

# ── Core prediction function ──────────────────────────────

def predict_vp(row, upsilon_fn, logL):
    """
    Compute predicted V_p for one galaxy given
    an upsilon function and log10(Lambda*).
    Returns (Vp, diagnostics_dict) or (None, None).
    """
    L36  = row['L36']
    MHI  = row['MHI']
    Reff = row['Reff']  if pd.notna(row['Reff'])  else np.nan
    Rdisk= row['Rdisk'] if pd.notna(row.get('Rdisk',np.nan)) \
                       else np.nan
    RHI  = row['RHI']   if pd.notna(row.get('RHI', np.nan)) \
                       else np.nan

    if pd.isna(Reff) or Reff <= 0:
        if pd.notna(Rdisk) and Rdisk > 0:
            Reff = Reff_to_h * Rdisk
        else:
            return None, None

    ups    = upsilon_fn(row)
    M_star = ups  * L36  * 1e9
    M_gas  = 1.33 * MHI  * 1e9
    h_star = Reff / Reff_to_h
    h_gas  = (RHI / RHI_to_h_gas
              if pd.notna(RHI) and RHI > 0
              else 2.5 * h_star)
    r_p    = 3.0 * Reff

    M_se   = M_star * enclosed_exp(r_p, h_star)
    M_ge   = M_gas  * enclosed_exp(r_p, h_gas)
    M_cone = max(M_se + (1.0/hz_base)*M_ge, 1e6)

    Lam    = 10**logL
    Vp     = (Lam**2 * G_kpc * M_cone * mu)**alpha

    diag = {
        'M_cone':    M_cone,
        'Sigma_bar': M_cone / (np.pi * r_p**2),
        'ups':       ups,
        'r_p':       r_p,
        'Reff':      Reff,
        'fgas':      M_gas/(M_star+M_gas),
    }
    return Vp, diag

# ── Metrics for any upsilon function ─────────────────────

def run_model(df, upsilon_fn, logL, label=''):
    resids = []
    vflat  = []
    reff   = []
    fgas   = []
    ups_v  = []
    sigma  = []
    T_v    = []

    for _, row in df.iterrows():
        Vobs = row['Vflat']
        if Vobs <= 0:
            continue
        Vp, diag = predict_vp(row, upsilon_fn, logL)
        if Vp is None or Vp <= 0:
            continue
        resid = np.log10(Vobs) - np.log10(Vp)
        resids.append(resid)
        vflat.append(Vobs)
        reff.append(diag['Reff'])
        fgas.append(diag['fgas'])
        ups_v.append(diag['ups'])
        sigma.append(diag['Sigma_bar'])
        T_v.append(row['T'] if pd.notna(row['T'])
                   else np.nan)

    arr  = np.array(resids)
    reff = np.array(reff)
    fgas = np.array(fgas)

    rV,pV = spearmanr(vflat, arr)
    rR,pR = spearmanr(reff,  arr)
    rG,_  = spearmanr(fgas,  arr)
    rU,pU = spearmanr(ups_v, arr)
    rS,_  = spearmanr(sigma, arr)

    T_arr   = np.array(T_v)
    T_clean = [(t,r) for t,r in zip(T_arr, arr)
               if np.isfinite(t)]
    rT      = (spearmanr([x[0] for x in T_clean],
                          [x[1] for x in T_clean])[0]
               if T_clean else np.nan)

    mae  = np.mean(np.abs(arr))
    bias = np.mean(arr)

    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    print(f"  N={len(arr)}  MAE={mae:.5f}  Bias={bias:+.5f}")
    print(f"  rho(resid, Vflat    ): {rV:+.4f}  p={pV:.3e}")
    print(f"  rho(resid, R_eff    ): {rR:+.4f}  p={pR:.3e}")
    print(f"  rho(resid, fgas     ): {rG:+.4f}")
    print(f"  rho(resid, Ups      ): {rU:+.4f}  p={pU:.3e}")
    print(f"  rho(resid, Sigma_bar): {rS:+.4f}")
    print(f"  rho(resid, T        ): {rT:+.4f}")
    print(f"  Ups: min={min(ups_v):.3f}  "
          f"max={max(ups_v):.3f}  "
          f"mean={np.mean(ups_v):.3f}")

    return {
        'MAE':  mae,  'Bias': bias,
        'rho_V': rV,  'p_V':  pV,
        'rho_R': rR,  'p_R':  pR,
        'rho_G': rG,  'rho_U': rU,
        'N':    len(arr),
        'resids': arr,
        'reff':   reff,
        'vflat':  np.array(vflat),
        'ups':    np.array(ups_v),
    }

# ── Optimise each model ───────────────────────────────────

print("Optimising models...\n")

# ── Model A: constant Upsilon ─────────────────────────────
res_A = minimize_scalar(
    lambda lL: run_model(
        dev,
        lambda row: upsilon_const(row),
        lL, '')['MAE'],
    bounds=(2.0, 3.0), method='bounded',
    options={'xatol':1e-4})
logL_A = res_A.x
mA = run_model(dev,
               lambda row: upsilon_const(row),
               logL_A,
               f'Model A — Constant Upsilon  '
               f'Lambda*={10**logL_A:.1f}')

# ── Model B: Upsilon(Sigma_star) ──────────────────────────
def obj_B(params):
    ups0, gamma, logL = params
    mae = run_model(
        dev,
        lambda row: upsilon_sigma(row, ups0, gamma),
        logL, '')['MAE']
    return mae

res_B = minimize(obj_B,
                 x0=[-0.3, 0.2, np.log10(278)],
                 method='Nelder-Mead',
                 options={'xatol':1e-5,
                          'fatol':1e-6,
                          'maxiter':5000})
ups0_B, gamma_B, logL_B = res_B.x
mB = run_model(
    dev,
    lambda row: upsilon_sigma(row, ups0_B, gamma_B),
    logL_B,
    f'Model B — Upsilon(Sigma_star)  '
    f'ups0={ups0_B:.3f}  gamma={gamma_B:.3f}  '
    f'Lambda*={10**logL_B:.1f}')

# ── Model C: Upsilon(SBeff) if column available ───────────
has_SBeff = ('SBeff' in dev.columns and
             dev['SBeff'].notna().sum() > 20)

if has_SBeff:
    def obj_C(params):
        a, b, logL = params
        return run_model(
            dev,
            lambda row: upsilon_SBeff(row, a, b),
            logL, '')['MAE']

    res_C = minimize(obj_C,
                     x0=[-0.5, 0.03,
                          np.log10(278)],
                     method='Nelder-Mead',
                     options={'xatol':1e-5,
                              'fatol':1e-6,
                              'maxiter':5000})
    a_C, b_C, logL_C = res_C.x
    mC = run_model(
        dev,
        lambda row: upsilon_SBeff(row, a_C, b_C),
        logL_C,
        f'Model C — Upsilon(SBeff)  '
        f'a={a_C:.3f}  b={b_C:.4f}  '
        f'Lambda*={10**logL_C:.1f}')
else:
    print("\nModel C skipped: SBeff not available.")
    mC = None

# ── Summary ───────────────────────────────────────────────
print("\n" + "="*60)
print("SUMMARY — ALL MODELS")
print("="*60)
print(f"  {'Model':<35} {'MAE':>8} "
      f"{'rhoV':>8} {'rhoReff':>9} {'N':>5}")
print(f"  {'-'*65}")

rows_s = [
    ('A: Constant Upsilon (baseline)', mA),
    ('B: Upsilon(Sigma_star)',          mB),
]
if mC:
    rows_s.append(('C: Upsilon(SBeff)', mC))

for name, m in rows_s:
    print(f"  {name:<35} {m['MAE']:>8.5f} "
          f"{m['rho_V']:>+8.4f} "
          f"{m['rho_R']:>+9.4f} "
          f"{m['N']:>5}")

# ── Implied Upsilon vs R_eff scatter ─────────────────────
print("\n--- Implied Upsilon by R_eff quartile ---")
print("(What Upsilon does each galaxy need to fit Vflat?)")
print("Using Model A Lambda* as reference.\n")

Lopt = 10**logL_A
implied_rows = []
for _, row in dev.iterrows():
    Vobs = row['Vflat']
    if Vobs <= 0: continue
    Reff = row['Reff'] if pd.notna(row['Reff']) else np.nan
    if pd.isna(Reff) or Reff <= 0: continue

    h_star = Reff / Reff_to_h
    RHI    = row['RHI'] if pd.notna(row.get('RHI',np.nan)) \
                       else np.nan
    h_gas  = (RHI/RHI_to_h_gas
              if pd.notna(RHI) and RHI > 0
              else 2.5*h_star)
    r_p    = 3.0 * Reff
    M_gas  = 1.33 * row['MHI'] * 1e9
    M_ge   = M_gas * enclosed_exp(r_p, h_gas)
    fstar  = enclosed_exp(r_p, h_star)
    L_enc  = row['L36'] * 1e9 * fstar

    M_need = (Vobs / (Lopt**2 * G_kpc * mu)**alpha)**(1/alpha)
    Ups_need = ((M_need - (1/hz_base)*M_ge)
                / max(L_enc, 1e3))

    if Ups_need <= 0 or Ups_need > 5:
        continue

    implied_rows.append({
        'Galaxy': row['Galaxy'],
        'R_eff':  Reff,
        'T':      row['T'] if pd.notna(row['T']) else np.nan,
        'Ups_const': upsilon_const(row),
        'Ups_need':  Ups_need,
        'logSstar':  np.log10(
            row['L36']/(2*np.pi*Reff**2)),
    })

di = pd.DataFrame(implied_rows)
di['Rq'] = pd.qcut(di['R_eff'], 4,
                   labels=['Q1 small','Q2',
                            'Q3','Q4 large'])
grp = di.groupby('Rq', observed=True).agg(
    N         = ('Ups_need',  'count'),
    R_eff_mean= ('R_eff',     'mean'),
    Ups_const = ('Ups_const', 'mean'),
    Ups_need  = ('Ups_need',  'mean'),
    Ups_std   = ('Ups_need',  'std'),
    logSstar  = ('logSstar',  'mean'),
).round(4)
print(grp.to_string())

rho, p = spearmanr(di['R_eff'], di['Ups_need'])
print(f"\n  rho(Ups_need, R_eff  ): {rho:+.4f}  p={p:.3e}")
rho, p = spearmanr(di['logSstar'], di['Ups_need'])
print(f"  rho(Ups_need, logS*  ): {rho:+.4f}  p={p:.3e}")
rho, p = spearmanr(di['T'],       di['Ups_need'])
print(f"  rho(Ups_need, T      ): {rho:+.4f}  p={p:.3e}")

Optimising models...


  
  N=135  MAE=0.06420  Bias=+0.03144
  rho(resid, Vflat    ): +0.0780  p=3.687e-01
  rho(resid, R_eff    ): -0.2494  p=3.528e-03
  rho(resid, fgas     ): +0.0656
  rho(resid, Ups      ): -0.0459  p=5.973e-01
  rho(resid, Sigma_bar): +0.0203
  rho(resid, T        ): +0.0570
  Ups: min=0.500  max=0.700  mean=0.524

  
  N=135  MAE=0.09370  Bias=-0.08660
  rho(resid, Vflat    ): +0.0780  p=3.687e-01
  rho(resid, R_eff    ): -0.2494  p=3.528e-03
  rho(resid, fgas     ): +0.0656
  rho(resid, Ups      ): -0.0459  p=5.973e-01
  rho(resid, Sigma_bar): +0.0203
  rho(resid, T        ): +0.0570
  Ups: min=0.500  max=0.700  mean=0.524

  
  N=135  MAE=0.11994  Bias=+0.10439
  rho(resid, Vflat    ): +0.0780  p=3.687e-01
  rho(resid, R_eff    ): -0.2494  p=3.528e-03
  rho(resid, fgas     ): +0.0656
  rho(resid, Ups      ): -0.0459  p=5.973e-01
  rho(resid, Sigma_bar): +0.0203
  rho(resid, T        ): +0.0570
  Ups: min=0.500  max=0.700  mean=0.524

  
  N=135  MAE=0.05677  B

/tmp/ipykernel_8249/3584411542.py:134: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rU,pU = spearmanr(ups_v, arr)
/tmp/ipykernel_8249/3584411542.py:48: RuntimeWarning: overflow encountered in scalar power
  ups = 10**(a + b * SBeff)



  
  N=135  MAE=0.06277  Bias=+0.00325
  rho(resid, Vflat    ): -0.1814  p=3.527e-02
  rho(resid, R_eff    ): -0.3905  p=2.841e-06
  rho(resid, fgas     ): +0.3454
  rho(resid, Ups      ): +nan  p=nan
  rho(resid, Sigma_bar): -0.2607
  rho(resid, T        ): +0.2708
  Ups: min=3.000  max=3.000  mean=3.000

  
  N=135  MAE=0.06298  Bias=+0.01218
  rho(resid, Vflat    ): -0.1814  p=3.527e-02
  rho(resid, R_eff    ): -0.3905  p=2.841e-06
  rho(resid, fgas     ): +0.3454
  rho(resid, Ups      ): +nan  p=nan
  rho(resid, Sigma_bar): -0.2607
  rho(resid, T        ): +0.2708
  Ups: min=3.000  max=3.000  mean=3.000

  
  N=135  MAE=0.06308  Bias=+0.01361
  rho(resid, Vflat    ): -0.1814  p=3.527e-02
  rho(resid, R_eff    ): -0.3905  p=2.841e-06
  rho(resid, fgas     ): +0.3454
  rho(resid, Ups      ): +nan  p=nan
  rho(resid, Sigma_bar): -0.2607
  rho(resid, T        ): +0.2708
  Ups: min=3.000  max=3.000  mean=3.000

  
  N=135  MAE=0.06289  Bias=+0.01039
  rho(resid, Vflat    ): -0.1814  p=

In [11]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize, minimize_scalar
from scipy.stats import spearmanr

G_kpc        = 4.30091e-6
Reff_to_h    = 1.678
RHI_to_h_gas = 3.5
hz_base      = 2.0
alpha        = 0.25
mu           = 0.0824

# Reference values fixed from prior run — do not refit
R_REF   = 3.9    # kpc, sample mean R_eff
UPS_REF = 0.52   # baseline constant Upsilon

def enclosed_exp(r, h):
    if pd.isna(h) or h <= 0 or r <= 0:
        return 1.0
    u = r / h
    return 1.0 - np.exp(-u) * (1.0 + u)

def upsilon_powerlaw(Reff, ups_ref, delta):
    """
    Upsilon = ups_ref * (R_eff / R_ref)^(-delta)
    delta > 0: compact galaxies get higher Upsilon
    delta = 0: recovers constant Upsilon
    Bounded to [0.1, 3.0] for physical safety.
    """
    if pd.isna(Reff) or Reff <= 0:
        return ups_ref
    ups = ups_ref * (Reff / R_REF)**(-delta)
    return float(np.clip(ups, 0.1, 3.0))

def predict_one(row, ups_ref, delta, logL):
    """Returns (Vp, diag) or (None, None)."""
    Reff  = row['Reff']  if pd.notna(row['Reff'])  else np.nan
    Rdisk = row['Rdisk'] if pd.notna(row.get('Rdisk', np.nan)) \
                        else np.nan
    RHI   = row['RHI']   if pd.notna(row.get('RHI',  np.nan)) \
                        else np.nan

    if pd.isna(Reff) or Reff <= 0:
        if pd.notna(Rdisk) and Rdisk > 0:
            Reff = Reff_to_h * Rdisk
        else:
            return None, None

    ups    = upsilon_powerlaw(Reff, ups_ref, delta)
    M_star = ups  * row['L36'] * 1e9
    M_gas  = 1.33 * row['MHI'] * 1e9
    h_star = Reff / Reff_to_h
    h_gas  = (RHI / RHI_to_h_gas
              if pd.notna(RHI) and RHI > 0
              else 2.5 * h_star)
    r_p    = 3.0 * Reff

    M_se   = M_star * enclosed_exp(r_p, h_star)
    M_ge   = M_gas  * enclosed_exp(r_p, h_gas)
    M_cone = max(M_se + M_ge / hz_base, 1e6)

    Vp = ((10**logL)**2 * G_kpc * M_cone * mu)**alpha

    return Vp, {
        'Reff': Reff, 'ups': ups,
        'M_cone': M_cone,
        'fgas': M_gas / (M_star + M_gas),
        'Sigma_bar': M_cone / (np.pi * r_p**2),
    }

def run_model(df, ups_ref, delta, logL,
              label='', verbose=True):
    resids = []
    vflat  = []
    reff   = []
    fgas   = []
    ups_v  = []
    sigma  = []
    T_v    = []

    for _, row in df.iterrows():
        Vobs = row['Vflat']
        if Vobs <= 0:
            continue
        Vp, diag = predict_one(row, ups_ref, delta, logL)
        if Vp is None or Vp <= 0:
            continue
        resids.append(np.log10(Vobs) - np.log10(Vp))
        vflat.append(Vobs)
        reff.append(diag['Reff'])
        fgas.append(diag['fgas'])
        ups_v.append(diag['ups'])
        sigma.append(diag['Sigma_bar'])
        T_v.append(row['T'] if pd.notna(row['T'])
                   else np.nan)

    arr   = np.array(resids)
    reff  = np.array(reff)
    fgas  = np.array(fgas)
    vflat = np.array(vflat)

    rV, pV = spearmanr(vflat, arr)
    rR, pR = spearmanr(reff,  arr)
    rG, _  = spearmanr(fgas,  arr)
    rS, _  = spearmanr(sigma, arr)
    T_arr  = np.array(T_v)
    T_c    = [(t,r) for t,r in zip(T_arr, arr)
              if np.isfinite(t)]
    rT     = (spearmanr([x[0] for x in T_c],
                         [x[1] for x in T_c])[0]
              if T_c else np.nan)
    rU, pU = spearmanr(ups_v, arr)

    mae  = np.mean(np.abs(arr))
    bias = np.mean(arr)

    if verbose:
        def sig(p):
            return ("***" if p<0.001 else
                    "**"  if p<0.01  else
                    "*"   if p<0.05  else
                    "."   if p<0.10  else " ")
        print(f"\n{'='*55}")
        print(f"  {label}")
        print(f"{'='*55}")
        print(f"  N={len(arr):<5} "
              f"MAE={mae:.5f}  Bias={bias:+.5f}")
        print(f"  Ups range: [{min(ups_v):.3f}, "
              f"{max(ups_v):.3f}]  "
              f"mean={np.mean(ups_v):.3f}")
        print(f"  rho(resid, Vflat    ): "
              f"{rV:+.4f}  p={pV:.3e}  {sig(pV)}")
        print(f"  rho(resid, R_eff    ): "
              f"{rR:+.4f}  p={pR:.3e}  {sig(pR)}")
        print(f"  rho(resid, fgas     ): "
              f"{rG:+.4f}")
        print(f"  rho(resid, Sigma_bar): "
              f"{rS:+.4f}")
        print(f"  rho(resid, T        ): "
              f"{rT:+.4f}")
        print(f"  rho(resid, Ups      ): "
              f"{rU:+.4f}  p={pU:.3e}  {sig(pU)}")

    return {
        'MAE':   mae,  'Bias':  bias,
        'rho_V': rV,   'p_V':   pV,
        'rho_R': rR,   'p_R':   pR,
        'rho_G': rG,
        'rho_U': rU,   'p_U':   pU,
        'N':     len(arr),
        'resids': arr,
        'reff':   reff,
        'vflat':  vflat,
        'ups':    np.array(ups_v),
    }

# ── Step 1: Establish clean baseline (delta=0) ───────────

print("STEP 1: BASELINE — constant Upsilon")
print("Pre-registered expectation:")
print("  MAE ≈ 0.057, rho(resid,R_eff) ≈ -0.25 p≈0.004")
print()

res_base = minimize_scalar(
    lambda lL: run_model(dev, UPS_REF, 0.0, lL,
                         verbose=False)['MAE'],
    bounds=(2.0, 3.0), method='bounded',
    options={'xatol': 1e-5})
logL_base = res_base.x
m_base = run_model(
    dev, UPS_REF, 0.0, logL_base,
    label=f'Baseline  ups={UPS_REF:.3f}  '
          f'delta=0  Lambda*={10**logL_base:.1f}')

# ── Step 2: Delta scan — Lambda* re-optimised each time ──

print("\n\nSTEP 2: DELTA SCAN")
print("Pre-registered prediction: MAE minimum at delta>0")
print("Pre-registered prediction: rho(resid,R_eff) → 0")
print()

print(f"{'delta':>7} {'ups_ref':>8} {'Lambda*':>9} "
      f"{'MAE':>9} {'Bias':>8} "
      f"{'rhoV':>7} {'rhoReff':>8} {'pReff':>10}")
print("-" * 75)

delta_grid = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25,
              0.30, 0.35, 0.40, 0.50, 0.60,
              0.75, 1.00]
scan_results = []

for delta in delta_grid:
    # Reoptimise Lambda* and ups_ref jointly at each delta
    def obj_scan(params):
        ups_r, lL = params
        if ups_r < 0.1 or ups_r > 2.0: return 999.
        if lL < 2.0 or lL > 3.0:       return 999.
        return run_model(dev, ups_r, delta, lL,
                         verbose=False)['MAE']

    r = minimize(obj_scan,
                 x0=[UPS_REF, logL_base],
                 method='Nelder-Mead',
                 options={'xatol':1e-5,
                          'fatol':1e-6,
                          'maxiter':3000})
    ur, lL = r.x
    ur     = float(np.clip(ur, 0.1, 2.0))
    m = run_model(dev, ur, delta, lL, verbose=False)
    scan_results.append({
        'delta': delta,
        'ups_r': ur,
        'logL':  lL,
        'm':     m,
    })
    mark = " ← baseline" if delta == 0.0 else ""
    print(f"{delta:>7.2f} {ur:>8.4f} {10**lL:>9.2f} "
          f"{m['MAE']:>9.5f} {m['Bias']:>+8.5f} "
          f"{m['rho_V']:>+7.4f} "
          f"{m['rho_R']:>+8.4f} "
          f"{m['p_R']:>10.3e}{mark}")

# ── Step 3: Full optimisation (ups_ref, delta, logL) ─────

print("\n\nSTEP 3: FULL OPTIMISATION (ups_ref, delta, Lambda*)")

best_scan = min(scan_results, key=lambda x: x['m']['MAE'])
x0 = [best_scan['ups_r'],
      best_scan['delta'],
      best_scan['logL']]

def obj_full(params):
    ur, d, lL = params
    if ur < 0.1 or ur > 2.0: return 999.
    if d  < 0.0 or d  > 2.0: return 999.
    if lL < 2.0 or lL > 3.0: return 999.
    return run_model(dev, ur, d, lL,
                     verbose=False)['MAE']

res_full = minimize(obj_full, x0=x0,
                    method='Nelder-Mead',
                    options={'xatol':1e-6,
                             'fatol':1e-7,
                             'maxiter':8000})
ur_opt, d_opt, lL_opt = res_full.x
ur_opt = float(np.clip(ur_opt, 0.1, 2.0))
d_opt  = max(d_opt, 0.0)

m_full = run_model(
    dev, ur_opt, d_opt, lL_opt,
    label=f'Optimised  '
          f'ups_ref={ur_opt:.4f}  '
          f'delta={d_opt:.4f}  '
          f'Lambda*={10**lL_opt:.2f}')

# ── Step 4: Implied Υ quartile check at optimum ──────────

print("\n\nSTEP 4: IMPLIED Υ BY R_EFF QUARTILE AT OPTIMUM")
print("If the correction is working, Q1 and Q4 should")
print("have Ups_model ≈ Ups_need after correction.\n")

Lopt = 10**lL_opt
imp  = []
for _, row in dev.iterrows():
    Vobs = row['Vflat']
    if Vobs <= 0: continue
    Reff = row['Reff'] if pd.notna(row['Reff']) else np.nan
    if pd.isna(Reff) or Reff <= 0: continue

    h_star = Reff / Reff_to_h
    RHI    = row['RHI'] if pd.notna(row.get('RHI', np.nan)) \
                       else np.nan
    h_gas  = (RHI/RHI_to_h_gas
              if pd.notna(RHI) and RHI > 0
              else 2.5*h_star)
    r_p    = 3.0 * Reff
    M_gas  = 1.33 * row['MHI'] * 1e9
    M_ge   = M_gas * enclosed_exp(r_p, h_gas)
    fstar  = enclosed_exp(r_p, h_star)
    L_enc  = row['L36'] * 1e9 * fstar

    M_need = (Vobs / (Lopt**2 * G_kpc * mu)**alpha
              )**(1.0/alpha)
    Ups_need = ((M_need - M_ge/hz_base)
                / max(L_enc, 1e3))
    if Ups_need <= 0 or Ups_need > 5:
        continue

    imp.append({
        'R_eff':     Reff,
        'Ups_model': upsilon_powerlaw(Reff, ur_opt, d_opt),
        'Ups_need':  Ups_need,
        'T':         row['T'] if pd.notna(row['T'])
                     else np.nan,
    })

di = pd.DataFrame(imp)
di['Rq'] = pd.qcut(di['R_eff'], 4,
                   labels=['Q1 small','Q2',
                            'Q3','Q4 large'])
grp = di.groupby('Rq', observed=True).agg(
    N          = ('R_eff',     'count'),
    R_eff_mean = ('R_eff',     'mean'),
    Ups_model  = ('Ups_model', 'mean'),
    Ups_need   = ('Ups_need',  'mean'),
    Ups_std    = ('Ups_need',  'std'),
).round(4)
grp['gap'] = (grp['Ups_need'] - grp['Ups_model']).round(4)
print(grp.to_string())
print()
print("  gap = Ups_need - Ups_model")
print("  gap ≈ 0 across quartiles → correction absorbed the trend")
print("  gap persists → size effect is not a pure Υ story")

rho_rn, p_rn = spearmanr(di['R_eff'], di['Ups_need'])
rho_rm, p_rm = spearmanr(di['R_eff'], di['Ups_model'])
print(f"\n  rho(Ups_need,  R_eff): {rho_rn:+.4f}  "
      f"p={p_rn:.3e}")
print(f"  rho(Ups_model, R_eff): {rho_rm:+.4f}  "
      f"p={p_rm:.3e}  "
      f"(should match Ups_need by construction)")

# ── Step 5: Summary ───────────────────────────────────────

print("\n\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"\n  {'Model':<42} "
      f"{'MAE':>8} {'rhoV':>8} {'rhoReff':>9}")
print(f"  {'-'*69}")
print(f"  {'Baseline (constant Ups, delta=0)':<42} "
      f"{m_base['MAE']:>8.5f} "
      f"{m_base['rho_V']:>+8.4f} "
      f"{m_base['rho_R']:>+9.4f}")
print(f"  {'Power-law Ups(Reff) optimised':<42} "
      f"{m_full['MAE']:>8.5f} "
      f"{m_full['rho_V']:>+8.4f} "
      f"{m_full['rho_R']:>+9.4f}")

delta_mae = m_full['MAE'] - m_base['MAE']
print(f"\n  MAE change: {delta_mae:+.5f} dex  "
      f"({'improvement' if delta_mae<0 else 'no improvement'})")
print(f"  delta_opt = {d_opt:.4f}")
print()
print("  PRE-REGISTERED VERDICT:")
print("  If delta_opt > 0.05 AND MAE drops > 0.003 AND")
print("  rho(resid,R_eff) loses significance:")
print("  → Size-dependent Upsilon is a real effect.")
print()
print("  If MAE does not drop OR rho(resid,R_eff) persists:")
print("  → The R_eff trend is dynamical, not a Υ artifact.")
print("  → The cone geometry or r_p definition is wrong.")

STEP 1: BASELINE — constant Upsilon
Pre-registered expectation:
  MAE ≈ 0.057, rho(resid,R_eff) ≈ -0.25 p≈0.004



/tmp/ipykernel_8249/1425464013.py:112: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rU, pU = spearmanr(ups_v, arr)



  Baseline  ups=0.520  delta=0  Lambda*=279.1
  N=135   MAE=0.05753  Bias=+0.00050
  Ups range: [0.520, 0.520]  mean=0.520
  rho(resid, Vflat    ): +0.1216  p=1.600e-01   
  rho(resid, R_eff    ): -0.2716  p=1.441e-03  **
  rho(resid, fgas     ): +0.0185
  rho(resid, Sigma_bar): +0.0928
  rho(resid, T        ): -0.0249
  rho(resid, Ups      ): +nan  p=nan   


STEP 2: DELTA SCAN
Pre-registered prediction: MAE minimum at delta>0
Pre-registered prediction: rho(resid,R_eff) → 0

  delta  ups_ref   Lambda*       MAE     Bias    rhoV  rhoReff      pReff
---------------------------------------------------------------------------
   0.00   0.4574    295.87   0.05752 -0.00173 +0.1586  -0.2559  2.733e-03 ← baseline
   0.05   0.5289    281.47   0.05690 -0.00333 +0.1382  -0.2385  5.334e-03
   0.10   0.4781    293.37   0.05645 -0.00467 +0.1876  -0.1923  2.548e-02
   0.15   0.4732    294.12   0.05605 -0.00504 +0.2082  -0.1570  6.891e-02
   0.20   0.4659    293.00   0.05570 -0.00360 +0.2308  -0.120

In [13]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize, minimize_scalar
from scipy.stats import spearmanr
from numpy.random import default_rng

# ═══════════════════════════════════════════════════════════
# PHYSICAL CONSTANTS AND FIXED MODEL PARAMETERS
# These are frozen from prior work and must not be refit
# on the blind subset under any circumstances.
# ═══════════════════════════════════════════════════════════

G_kpc        = 4.30091e-6   # kpc M_sun^-1 (km/s)^2
Reff_to_h    = 1.678        # R_eff = 1.678 * h_disk
RHI_to_h_gas = 3.5         # R_HI  = 3.5   * h_gas
hz_base      = 2.0          # gas vertical factor
alpha        = 0.25         # V ~ M^(1/4)
mu           = 0.0824       # TAFA coupling constant
R_REF        = 3.9          # kpc, sample mean R_eff (frozen)
UPS_CONST    = 0.52         # constant Upsilon (frozen)

# ═══════════════════════════════════════════════════════════
# CALIBRATION / BLIND SPLIT
# Seed is pre-registered. Do not change after first run.
# 70 galaxies calibration, 65 galaxies blind.
# ═══════════════════════════════════════════════════════════

SPLIT_SEED = 20240117
CAL_FRAC   = 70   # exact count, not fraction

def make_split(df, n_cal=CAL_FRAC, seed=SPLIT_SEED):
    """
    Stratified split on log(Vflat) quartile so that
    the calibration set covers the full dynamic range.
    Returns (cal_df, blind_df).
    """
    rng = default_rng(seed)
    df  = df.copy().reset_index(drop=True)

    # Quartile labels on Vflat
    df['_vq'] = pd.qcut(df['Vflat'], 4,
                        labels=False,
                        duplicates='drop')
    cal_idx = []
    for q in df['_vq'].unique():
        grp = df[df['_vq'] == q].index.tolist()
        rng.shuffle(grp)
        n_take = max(1, round(n_cal * len(grp) / len(df)))
        cal_idx.extend(grp[:n_take])

    # Trim or pad to exactly n_cal
    cal_idx = sorted(cal_idx)
    if len(cal_idx) > n_cal:
        extra  = rng.choice(
            cal_idx, len(cal_idx) - n_cal,
            replace=False).tolist()
        cal_idx = [i for i in cal_idx
                   if i not in extra]
    elif len(cal_idx) < n_cal:
        remaining = [i for i in df.index
                     if i not in cal_idx]
        rng.shuffle(remaining)
        cal_idx.extend(
            remaining[:n_cal - len(cal_idx)])

    blind_idx = [i for i in df.index
                 if i not in cal_idx]

    cal   = df.loc[sorted(cal_idx)].drop(
        columns='_vq').reset_index(drop=True)
    blind = df.loc[sorted(blind_idx)].drop(
        columns='_vq').reset_index(drop=True)
    return cal, blind

# ═══════════════════════════════════════════════════════════
# GEOMETRY
# ═══════════════════════════════════════════════════════════

def enclosed_exp(r, h):
    if pd.isna(h) or h <= 0 or r <= 0:
        return 1.0
    u = r / h
    return 1.0 - np.exp(-u) * (1.0 + u)

def get_geometry(row):
    """
    Returns (Reff, h_star, h_gas, r_p, RHI) or None.
    """
    Reff  = row['Reff']  if pd.notna(row['Reff'])  else np.nan
    Rdisk = row['Rdisk'] if pd.notna(
        row.get('Rdisk', np.nan)) else np.nan
    RHI   = row['RHI']   if pd.notna(
        row.get('RHI',   np.nan)) else np.nan

    if pd.isna(Reff) or Reff <= 0:
        if pd.notna(Rdisk) and Rdisk > 0:
            Reff = Reff_to_h * Rdisk
        else:
            return None

    h_star = Reff / Reff_to_h
    h_gas  = (RHI / RHI_to_h_gas
              if pd.notna(RHI) and RHI > 0
              else 2.5 * h_star)
    r_p    = 3.0 * Reff
    return Reff, h_star, h_gas, r_p, RHI

# ═══════════════════════════════════════════════════════════
# UPSILON MODELS  (competing explanations for R_eff signal)
# ═══════════════════════════════════════════════════════════

def ups_constant(row, ups=UPS_CONST):
    return ups

def ups_powerlaw_Reff(row, ups_ref, delta):
    """Ups = ups_ref * (R_eff / R_REF)^(-delta)"""
    geo = get_geometry(row)
    if geo is None: return ups_ref
    Reff = geo[0]
    return float(np.clip(
        ups_ref * (Reff / R_REF)**(-delta), 0.1, 3.0))

def ups_sigma_star(row, ups_ref, gamma, sig_ref):
    """
    Ups = ups_ref * (Sigma_star / sig_ref)^gamma
    Sigma_star = L36 / (2 pi Reff^2)
    """
    geo = get_geometry(row)
    if geo is None: return ups_ref
    Reff = geo[0]
    L36  = row['L36']
    if L36 <= 0: return ups_ref
    SigS = L36 / (2.0 * np.pi * Reff**2)
    return float(np.clip(
        ups_ref * (SigS / sig_ref)**gamma, 0.1, 3.0))

# ═══════════════════════════════════════════════════════════
# LAMBDA_EFF MODELS  (EFT running coupling predictions)
# ═══════════════════════════════════════════════════════════

def Leff_constant(row, logL0, ups_fn, ups_kw):
    """Fixed Lambda* — baseline TAFA."""
    return 10**logL0

def Leff_mbar_reff(row, logL0, delta_eft, ups_fn, ups_kw):
    """
    EFT physically-motivated form:
    Lambda_eff = Lambda0 * (1 + delta * M_bar/R_eff^2)^(1/2)
    M_bar in M_sun, R_eff in kpc.
    delta_eft has units kpc^2 / M_sun.
    """
    geo = get_geometry(row)
    if geo is None: return 10**logL0
    Reff = geo[0]
    ups  = ups_fn(row, **ups_kw)
    Mstar = ups * row['L36'] * 1e9
    Mgas  = 1.33 * row['MHI'] * 1e9
    Mbar  = Mstar + Mgas
    arg   = 1.0 + delta_eft * Mbar / Reff**2
    arg   = max(arg, 0.01)   # physical floor
    return 10**logL0 * np.sqrt(arg)

def Leff_proxy_reff(row, logL0, delta_prx, R0,
                    ups_fn, ups_kw):
    """
    Empirical proxy form:
    Lambda_eff = Lambda0 * (1 + delta * R0/R_eff)^(1/2)
    Described as proxy, not microscopic law.
    """
    geo = get_geometry(row)
    if geo is None: return 10**logL0
    Reff = geo[0]
    arg  = 1.0 + delta_prx * R0 / Reff
    arg  = max(arg, 0.01)
    return 10**logL0 * np.sqrt(arg)

# ═══════════════════════════════════════════════════════════
# CORE PREDICTOR
# ═══════════════════════════════════════════════════════════

def predict_vp(row, Leff_fn, Leff_kw,
               ups_fn, ups_kw):
    geo = get_geometry(row)
    if geo is None: return None, None
    Reff, h_star, h_gas, r_p, _ = geo

    Lam   = Leff_fn(row,
                    ups_fn=ups_fn,
                    ups_kw=ups_kw,
                    **Leff_kw)
    ups   = ups_fn(row, **ups_kw)
    Mstar = ups  * row['L36'] * 1e9
    Mgas  = 1.33 * row['MHI'] * 1e9

    M_se  = Mstar * enclosed_exp(r_p, h_star)
    M_ge  = Mgas  * enclosed_exp(r_p, h_gas)
    Mcone = max(M_se + M_ge / hz_base, 1e6)

    Vp    = (Lam**2 * G_kpc * Mcone * mu)**alpha

    return Vp, {
        'Reff':  Reff,
        'ups':   ups,
        'Lam':   Lam,
        'Mbar':  Mstar + Mgas,
        'Mcone': Mcone,
        'SigBar':Mcone / (np.pi * r_p**2),
        'fgas':  Mgas  / (Mstar + Mgas),
        'SigS': (row['L36'] / (2*np.pi*Reff**2)
                 if row['L36'] > 0 else np.nan),
    }

# ═══════════════════════════════════════════════════════════
# EVALUATION ENGINE
# ═══════════════════════════════════════════════════════════

def evaluate(df, Leff_fn, Leff_kw,
             ups_fn, ups_kw,
             label='', verbose=True):
    resids = []
    vflat  = []
    reff   = []
    fgas   = []
    lam    = []
    sigS   = []
    T_v    = []
    sigBar = []
    mbar   = []

    for _, row in df.iterrows():
        Vobs = row['Vflat']
        if Vobs <= 0: continue
        Vp, diag = predict_vp(
            row, Leff_fn, Leff_kw,
            ups_fn, ups_kw)
        if Vp is None or Vp <= 0: continue

        resids.append(np.log10(Vobs) - np.log10(Vp))
        vflat.append(Vobs)
        reff.append(diag['Reff'])
        fgas.append(diag['fgas'])
        lam.append(diag['Lam'])
        sigS.append(np.log10(diag['SigS'])
                    if diag['SigS'] > 0 else np.nan)
        T_v.append(row['T'] if pd.notna(row['T'])
                   else np.nan)
        sigBar.append(diag['SigBar'])
        mbar.append(diag['Mbar'])

    arr = np.array(resids)
    mae  = np.mean(np.abs(arr))
    bias = np.mean(arr)

    def rsp(x):
        x = np.array(x, dtype=float)
        m = np.isfinite(x) & np.isfinite(arr)
        if m.sum() < 5: return np.nan, np.nan
        return spearmanr(x[m], arr[m])

    rV,pV   = rsp(vflat)
    rR,pR   = rsp(reff)
    rG,_    = rsp(fgas)
    rSS,pSS = rsp(sigS)
    rT,_    = rsp(T_v)
    rSB,_   = rsp(sigBar)
    rMB,_   = rsp(mbar)

    def sig(p):
        if np.isnan(p): return "?"
        return ("***" if p<0.001 else "**" if p<0.01
                else "*" if p<0.05 else "." if p<0.10
                else " ")

    if verbose:
        print(f"\n{'═'*58}")
        print(f"  {label}")
        print(f"{'═'*58}")
        print(f"  N={len(arr):<5}  "
              f"MAE={mae:.5f}  Bias={bias:+.6f}")
        lv = np.array(lam)
        print(f"  Lambda_eff: [{lv.min():.1f}, "
              f"{lv.max():.1f}]  mean={lv.mean():.1f}")
        ups_v = [ups_fn(row, **ups_kw)
                 for _, row in df.iterrows()
                 if row['Vflat'] > 0]
        if ups_v:
            print(f"  Upsilon:  [{min(ups_v):.3f}, "
                  f"{max(ups_v):.3f}]  "
                  f"mean={np.mean(ups_v):.3f}")
        print(f"  rho(resid, Vflat    ): "
              f"{rV:+.4f}  p={pV:.3e}  {sig(pV)}")
        print(f"  rho(resid, R_eff    ): "
              f"{rR:+.4f}  p={pR:.3e}  {sig(pR)}")
        print(f"  rho(resid, logSigS* ): "
              f"{rSS:+.4f}  p={pSS:.3e}  {sig(pSS)}")
        print(f"  rho(resid, fgas     ): "
              f"{rG:+.4f}")
        print(f"  rho(resid, Sigma_bar): "
              f"{rSB:+.4f}")
        print(f"  rho(resid, logMbar  ): "
              f"{rMB:+.4f}")
        print(f"  rho(resid, T        ): "
              f"{rT:+.4f}")

    return {
        'MAE':mae,'Bias':bias,
        'rho_V':rV,'p_V':pV,
        'rho_R':rR,'p_R':pR,
        'rho_SS':rSS,'p_SS':pSS,
        'rho_G':rG,'rho_T':rT,
        'N':len(arr),
        'resids':arr,'reff':np.array(reff),
        'vflat':np.array(vflat),
    }

def obj(df, Leff_fn, Leff_kw,
        ups_fn, ups_kw):
    return evaluate(df, Leff_fn, Leff_kw,
                    ups_fn, ups_kw,
                    verbose=False)['MAE']

# ═══════════════════════════════════════════════════════════
# MAKE THE SPLIT
# ═══════════════════════════════════════════════════════════

cal, blind = make_split(dev)

sig_vals = [row['L36'] / (2*np.pi*row['Reff']**2)
            for _, row in cal.iterrows()
            if pd.notna(row['Reff'])
            and row['Reff'] > 0
            and row['L36'] > 0]
SIG_REF = float(np.median(sig_vals))

print("PRE-REGISTERED SPLIT")
print(f"  Calibration : N={len(cal)}")
print(f"  Blind       : N={len(blind)}")
print(f"  Split seed  : {SPLIT_SEED}")
print(f"  Sigma* ref  : {SIG_REF:.4f}  "
      f"[1e9 L_sun/kpc^2]")
print()
print("PARAMETERS FROZEN FROM PRIOR WORK:")
print(f"  R_REF       = {R_REF} kpc")
print(f"  UPS_CONST   = {UPS_CONST}")
print(f"  alpha       = {alpha}")
print(f"  mu          = {mu}")
print(f"  hz_base     = {hz_base}")
print()
print("THE BLIND SET WILL NOT BE TOUCHED UNTIL")
print("ALL CALIBRATION FITS ARE COMPLETE.")

# ═══════════════════════════════════════════════════════════
# CALIBRATION FITS — all parameter choices happen here only
# ═══════════════════════════════════════════════════════════

print("\n\n" + "═"*58)
print("CALIBRATION FITS")
print("═"*58)

# ── M0: Baseline — fixed Lambda, constant Ups ─────────────
res0 = minimize_scalar(
    lambda lL: obj(cal,
                   Leff_constant,
                   {'logL0': lL},
                   ups_constant, {}),
    bounds=(2.2, 2.8), method='bounded',
    options={'xatol': 1e-5})
logL0_M0 = res0.x
kw_M0_L  = {'logL0': logL0_M0}
kw_M0_U  = {}

m0_cal = evaluate(cal,
                  Leff_constant, kw_M0_L,
                  ups_constant,  kw_M0_U,
                  label=f'M0 (cal): Baseline  '
                        f'Lambda*={10**logL0_M0:.1f}')

# ── M1: Ups(Reff) running, fixed Lambda ──────────────────
# Stellar-population explanation for R_eff signal.
def fit_M1():
    def o(p):
        ur, d, lL = p
        if ur < 0.1 or ur > 2: return 999.
        if d  < 0.0:            return 999.
        if lL < 2.2 or lL>2.8: return 999.
        return obj(cal,
                   Leff_constant,
                   {'logL0': lL},
                   ups_powerlaw_Reff,
                   {'ups_ref': ur, 'delta': d})
    r = minimize(o, x0=[0.48, 0.44, logL0_M0],
                 method='Nelder-Mead',
                 options={'xatol':1e-6,'fatol':1e-7,
                          'maxiter':5000})
    return r.x

ur1, d1, lL1 = fit_M1()
kw_M1_L = {'logL0': lL1}
kw_M1_U = {'ups_ref': ur1, 'delta': max(d1, 0)}

m1_cal = evaluate(cal,
                  Leff_constant, kw_M1_L,
                  ups_powerlaw_Reff, kw_M1_U,
                  label=f'M1 (cal): Ups(R_eff)  '
                        f'ups_ref={ur1:.4f}  '
                        f'delta={max(d1,0):.4f}  '
                        f'Lambda*={10**lL1:.1f}')

# ── M2: EFT physically-motivated Lambda running ───────────
# Competing explanation: dynamical coupling, not Ups.
# Ups held constant at UPS_CONST.
def fit_M2():
    # delta_eft in kpc^2 / M_sun — very small number
    # Normalise: delta_eft = exp(x) * 1e-20
    def o(p):
        log_delt, lL = p
        if lL < 2.2 or lL > 2.8: return 999.
        d = np.exp(log_delt) * 1e-20
        if d < 0: return 999.
        return obj(cal,
                   Leff_mbar_reff,
                   {'logL0': lL,
                    'delta_eft': d},
                   ups_constant, {})
    r = minimize(o, x0=[0.0, logL0_M0],
                 method='Nelder-Mead',
                 options={'xatol':1e-6,'fatol':1e-7,
                          'maxiter':5000})
    log_d, lL2 = r.x
    return np.exp(log_d) * 1e-20, lL2

delt2, lL2 = fit_M2()
kw_M2_L = {'logL0': lL2, 'delta_eft': delt2}
kw_M2_U = {}

m2_cal = evaluate(cal,
                  Leff_mbar_reff, kw_M2_L,
                  ups_constant,   kw_M2_U,
                  label=f'M2 (cal): EFT Lambda(Mbar/Reff^2)  '
                        f'delta={delt2:.3e}  '
                        f'Lambda0={10**lL2:.1f}')

# ── M3: Proxy Lambda(R_eff), constant Ups ─────────────────
# Simpler empirical proxy form from EFT document §3.2.
def fit_M3():
    def o(p):
        d, R0, lL = p
        if d  < 0.0:            return 999.
        if R0 < 0.1:            return 999.
        if lL < 2.2 or lL>2.8: return 999.
        return obj(cal,
                   Leff_proxy_reff,
                   {'logL0':    lL,
                    'delta_prx':d,
                    'R0':       R0},
                   ups_constant, {})
    r = minimize(o, x0=[0.5, R_REF, logL0_M0],
                 method='Nelder-Mead',
                 options={'xatol':1e-6,'fatol':1e-7,
                          'maxiter':5000})
    d3, R0_3, lL3 = r.x
    return max(d3,0), max(R0_3,0.1), lL3

d3, R0_3, lL3 = fit_M3()
kw_M3_L = {'logL0':lL3,'delta_prx':d3,'R0':R0_3}
kw_M3_U = {}

m3_cal = evaluate(cal,
                  Leff_proxy_reff, kw_M3_L,
                  ups_constant,    kw_M3_U,
                  label=f'M3 (cal): Proxy Lambda(R0/Reff)  '
                        f'delta={d3:.4f}  R0={R0_3:.2f}  '
                        f'Lambda0={10**lL3:.1f}')

# ── M4: Joint — EFT Lambda AND Ups(Sigma_star) ────────────
# Maximum-flexibility model. Tests whether both corrections
# are needed simultaneously.
def fit_M4():
    def o(p):
        log_d, lL, ur, gam = p
        if lL < 2.2 or lL > 2.8: return 999.
        if ur < 0.1 or ur > 2.0: return 999.
        d = np.exp(log_d) * 1e-20
        if d < 0: return 999.
        return obj(cal,
                   Leff_mbar_reff,
                   {'logL0':lL,'delta_eft':d},
                   ups_sigma_star,
                   {'ups_ref':ur,'gamma':gam,
                    'sig_ref':SIG_REF})
    r = minimize(o, x0=[0.0, logL0_M0,
                         0.52, 0.15],
                 method='Nelder-Mead',
                 options={'xatol':1e-6,'fatol':1e-7,
                          'maxiter':8000})
    log_d,lL4,ur4,gam4 = r.x
    return np.exp(log_d)*1e-20, lL4, ur4, gam4

delt4, lL4, ur4, gam4 = fit_M4()
kw_M4_L = {'logL0':lL4,'delta_eft':delt4}
kw_M4_U = {'ups_ref':ur4,'gamma':gam4,
            'sig_ref':SIG_REF}

m4_cal = evaluate(cal,
                  Leff_mbar_reff, kw_M4_L,
                  ups_sigma_star, kw_M4_U,
                  label=f'M4 (cal): EFT Lambda + Ups(SigS*)  '
                        f'delta={delt4:.3e}  '
                        f'ups_ref={ur4:.4f}  '
                        f'gamma={gam4:.4f}')

# ═══════════════════════════════════════════════════════════
# CALIBRATION SUMMARY BEFORE BLIND TEST
# ═══════════════════════════════════════════════════════════

print("\n\n" + "═"*70)
print("CALIBRATION SUMMARY")
print("(All parameter choices frozen after this point)")
print("═"*70)
print(f"\n  {'Model':<45} "
      f"{'MAE':>8} {'rhoV':>8} {'rhoR':>8} {'rhoSS':>8}")
print(f"  {'-'*77}")

cal_results = [
    ('M0: Baseline  fixed-Lambda, const-Ups', m0_cal,
     kw_M0_L, kw_M0_U, Leff_constant,   ups_constant),
    ('M1: Ups(R_eff)  running Upsilon only',  m1_cal,
     kw_M1_L, kw_M1_U, Leff_constant,   ups_powerlaw_Reff),
    ('M2: EFT Lambda(Mbar/Reff^2)  const-Ups',m2_cal,
     kw_M2_L, kw_M2_U, Leff_mbar_reff,  ups_constant),
    ('M3: Proxy Lambda(R0/Reff)  const-Ups',  m3_cal,
     kw_M3_L, kw_M3_U, Leff_proxy_reff, ups_constant),
    ('M4: EFT Lambda + Ups(Sigma*)',           m4_cal,
     kw_M4_L, kw_M4_U, Leff_mbar_reff,  ups_sigma_star),
]

for name, m, *_ in cal_results:
    print(f"  {name:<45} {m['MAE']:>8.5f} "
          f"{m['rho_V']:>+8.4f} "
          f"{m['rho_R']:>+8.4f} "
          f"{m['rho_SS']:>+8.4f}")

# ═══════════════════════════════════════════════════════════
# BLIND TEST — parameters not touched
# ═══════════════════════════════════════════════════════════

print("\n\n" + "═"*70)
print("BLIND TEST  (N={})".format(len(blind)))
print("Parameters are exactly as calibrated above.")
print("No retuning permitted after this point.")
print("═"*70)

print(f"\n  {'Model':<45} "
      f"{'MAE':>8} {'rhoV':>8} {'rhoR':>8} {'rhoSS':>8}")
print(f"  {'-'*77}")

blind_results = []
for name, m_c, Lkw, Ukw, Lfn, Ufn in cal_results:
    mb = evaluate(blind, Lfn, Lkw, Ufn, Ukw,
                  verbose=False)
    blind_results.append((name, m_c, mb))
    print(f"  {name:<45} {mb['MAE']:>8.5f} "
          f"{mb['rho_V']:>+8.4f} "
          f"{mb['rho_R']:>+8.4f} "
          f"{mb['rho_SS']:>+8.4f}")

# ═══════════════════════════════════════════════════════════
# VERDICT
# ═══════════════════════════════════════════════════════════

print("\n\n" + "═"*70)
print("VERDICT TABLE")
print("═"*70)
print()
print("  Criterion for a model to beat the baseline:")
print("  1. Blind MAE < M0 blind MAE")
print("  2. No new significant correlation introduced")
print("     (all rho p-values > 0.05 on blind set)")
print("  3. Calibration-to-blind MAE degradation < 0.005 dex")
print("     (tests for overfitting)")
print()

m0_blind_mae = blind_results[0][2]['MAE']

print(f"  M0 blind MAE (reference) = {m0_blind_mae:.5f}")
print()
print(f"  {'Model':<45} {'BlindMAE':>9} "
      f"{'vs M0':>8} {'Overfit':>8} {'Clean':>6}")
print(f"  {'-'*77}")

for name, m_c, m_b in blind_results:
    delta_vs_base = m_b['MAE'] - m0_blind_mae
    overfit       = m_b['MAE'] - m_c['MAE']
    clean = (m_b['p_V']  > 0.05 and
             m_b['p_R']  > 0.05 and
             m_b['p_SS'] > 0.05)
    wins  = (m_b['MAE'] < m0_blind_mae and
             clean and
             overfit < 0.005)
    marker = " ✓ WINS" if wins else ""
    print(f"  {name:<45} "
          f"{m_b['MAE']:>9.5f} "
          f"{delta_vs_base:>+8.5f} "
          f"{overfit:>+8.5f} "
          f"{'Y' if clean else 'N':>6}"
          f"{marker}")

print()
print("  INTERPRETATION KEY:")
print("  vs M0    < 0:      model beats baseline on blind set")
print("  Overfit  < 0.005:  negligible calibration leak")
print("  Clean    = Y:      no residual systematics p<0.05")
print("  WINS requires all three simultaneously")
print()
print("  EFT VERDICT:")
print("  If M2 or M3 WINS: running Lambda_eff has")
print("    empirical support. Proceed to paper.")
print("  If M1 WINS but M2/M3 do not: the signal is")
print("    stellar population, not dynamical coupling.")
print("  If M4 WINS but neither M1 nor M2 alone does:")
print("    both effects are real and inseparable.")
print("  If only M0 survives: no correction is justified.")
print("    The R_eff systematic is a known limitation,")
print("    not a solvable problem with current data.")

PRE-REGISTERED SPLIT
  Calibration : N=70
  Blind       : N=65
  Split seed  : 20240117
  Sigma* ref  : 0.1711  [1e9 L_sun/kpc^2]

PARAMETERS FROZEN FROM PRIOR WORK:
  R_REF       = 3.9 kpc
  UPS_CONST   = 0.52
  alpha       = 0.25
  mu          = 0.0824
  hz_base     = 2.0

THE BLIND SET WILL NOT BE TOUCHED UNTIL
ALL CALIBRATION FITS ARE COMPLETE.


══════════════════════════════════════════════════════════
CALIBRATION FITS
══════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════
  M0 (cal): Baseline  Lambda*=284.1
══════════════════════════════════════════════════════════
  N=70     MAE=0.05640  Bias=-0.000957
  Lambda_eff: [284.1, 284.1]  mean=284.1
  Upsilon:  [0.520, 0.520]  mean=0.520
  rho(resid, Vflat    ): +0.2004  p=9.626e-02  .
  rho(resid, R_eff    ): -0.1989  p=9.879e-02  .
  rho(resid, logSigS* ): +0.1728  p=1.525e-01   
  rho(resid, fgas     ): -0.1434
  rho(resid, Sigma_bar): +0.1847
  rho(resid, logMbar  ): 

In [14]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr

# ── Recompute blind residuals for M0, M1, M3 ─────────────

def get_blind_residuals(Leff_fn, Leff_kw,
                        ups_fn,  ups_kw):
    rows = []
    for _, row in blind.iterrows():
        Vobs = row['Vflat']
        if Vobs <= 0: continue
        Vp, diag = predict_vp(
            row, Leff_fn, Leff_kw,
            ups_fn, ups_kw)
        if Vp is None or Vp <= 0: continue
        rows.append({
            'resid': np.log10(Vobs) - np.log10(Vp),
            'Vflat': Vobs,
            'Reff':  diag['Reff'],
            'Mbar':  diag['Mbar'],
            'fgas':  diag['fgas'],
            'Leff':  diag['Lam'],
            'T':     row['T'] if pd.notna(row['T'])
                     else np.nan,
        })
    return pd.DataFrame(rows)

d_M0 = get_blind_residuals(
    Leff_constant,   kw_M0_L,
    ups_constant,    kw_M0_U)
d_M1 = get_blind_residuals(
    Leff_constant,   kw_M1_L,
    ups_powerlaw_Reff, kw_M1_U)
d_M3 = get_blind_residuals(
    Leff_proxy_reff, kw_M3_L,
    ups_constant,    kw_M3_U)

# ── Test 1: Is M3 Vflat correlation from dwarfs only? ────

print("M3 BLIND: Vflat correlation by subsample")
print("-"*50)
for label, mask in [
    ("All         ", np.ones(len(d_M3), bool)),
    ("Dwarfs  <90 ", d_M3['Vflat'] < 90),
    ("Spirals ≥90 ", d_M3['Vflat'] >= 90),
]:
    sub = d_M3[mask]
    if len(sub) < 5: continue
    r, p = spearmanr(sub['Vflat'], sub['resid'])
    def sig(p):
        return ("***" if p<0.001 else "**" if p<0.01
                else "*" if p<0.05 else "." if p<0.10
                else " ")
    print(f"  {label}  N={len(sub):3d}  "
          f"rho={r:+.4f}  p={p:.3e}  {sig(p)}")

# ── Test 2: Partial correlation controlling for Mbar ─────
# If M3 Vflat correlation vanishes when Mbar is controlled,
# M3 is encoding TF. If it persists, something else is happening.

print("\nM3 BLIND: Partial rho(resid, Vflat | logMbar)")
print("-"*50)
for name, df_r in [('M0', d_M0),
                   ('M1', d_M1),
                   ('M3', d_M3)]:
    mask = (np.isfinite(df_r['Vflat']) &
            np.isfinite(df_r['Mbar'])  &
            np.isfinite(df_r['resid']))
    sub  = df_r[mask].copy()
    sub['logV'] = np.log10(sub['Vflat'])
    sub['logM'] = np.log10(sub['Mbar'])
    # Partial: residualise Vflat on Mbar first
    from numpy.polynomial import polynomial as P
    c = np.polyfit(sub['logM'], sub['logV'], 1)
    vflat_resid = sub['logV'] - np.polyval(c, sub['logM'])
    r,  p  = spearmanr(sub['logV'],    sub['resid'])
    rp, pp = spearmanr(vflat_resid,    sub['resid'])
    print(f"  {name}: raw rho={r:+.4f} p={p:.3e}  "
          f"partial rho={rp:+.4f} p={pp:.3e}")

# ── Test 3: Does M1 residual still have any structure? ───

print("\nM1 BLIND: Remaining structure check")
print("-"*50)
checks = [
    ('Vflat',     d_M1['Vflat']),
    ('Reff',      d_M1['Reff']),
    ('Mbar',      d_M1['Mbar']),
    ('fgas',      d_M1['fgas']),
    ('T',         d_M1['T']),
    ('logMbar',   np.log10(d_M1['Mbar'])),
    ('logReff',   np.log10(d_M1['Reff'])),
    ('Mbar/Reff2',d_M1['Mbar'] /
                  d_M1['Reff']**2),
]
for varname, x in checks:
    xa  = np.array(x, dtype=float)
    res = np.array(d_M1['resid'])
    mask = np.isfinite(xa) & np.isfinite(res)
    if mask.sum() < 5: continue
    r, p = spearmanr(xa[mask], res[mask])
    def sig(p):
        return ("***" if p<0.001 else "**" if p<0.01
                else "*" if p<0.05 else "." if p<0.10
                else " ")
    print(f"  rho(resid, {varname:<12}): "
          f"{r:+.4f}  p={p:.3e}  {sig(p)}")

# ── Test 4: M1 implied Upsilon vs known stellar pops ─────
# If Upsilon ~ (R_eff)^(-0.46) is a stellar population
# effect, it should correlate with morphological type T.

print("\nM1: Upsilon vs morphological type")
print("(If stellar population: later type = lower Ups)")
print("-"*50)
ups_v = []
T_v   = []
for _, row in blind.iterrows():
    if pd.isna(row.get('T', np.nan)): continue
    geo = get_geometry(row)
    if geo is None: continue
    ups = ups_powerlaw_Reff(row, **kw_M1_U)
    ups_v.append(ups)
    T_v.append(row['T'])

r, p = spearmanr(T_v, ups_v)
def sig(p):
    return ("***" if p<0.001 else "**" if p<0.01
            else "*" if p<0.05 else "." if p<0.10
            else " ")
print(f"  rho(Ups_M1, T): {r:+.4f}  p={p:.3e}  {sig(p)}")
print(f"  N = {len(T_v)}")
print()
print("  Expectation if stellar population:")
print("  Early type (low T) → higher Ups → rho < 0")
print("  Later type (high T) → lower Ups → rho < 0")
print("  Observed sign and significance:")
if p < 0.05 and r < 0:
    print("  ✓ Consistent with stellar population gradient")
elif p < 0.05 and r > 0:
    print("  ✗ Wrong sign — inconsistent with stellar pops")
else:
    print("  ~ No significant correlation with T")
    print("  ~ Stellar population interpretation not confirmed")
    print("    but also not ruled out (T is a crude proxy)")

# ── Summary ───────────────────────────────────────────────

print("\n\n" + "═"*60)
print("DIAGNOSTIC SUMMARY")
print("═"*60)
print("""
  The three tests above answer:

  Test 2 (partial rho):
    If M3 rho(resid,Vflat) vanishes after
    controlling for Mbar → M3 is encoding TF.
    If it persists → something physical.

  Test 3 (M1 structure):
    If M1 blind residuals are clean on ALL
    variables including Mbar/Reff^2 → the
    EFT variable has no residual signal.
    If Mbar/Reff^2 shows up → EFT mechanism
    may contribute independently of Ups.

  Test 4 (Ups vs T):
    If Ups(R_eff) anti-correlates with T →
    the correction is absorbing a real
    stellar population gradient.
    If not → the Ups(R_eff) form is a proxy
    for something else and the physical
    interpretation needs revision.
""")

M3 BLIND: Vflat correlation by subsample
--------------------------------------------------
  All           N= 65  rho=+0.2526  p=4.233e-02  *
  Dwarfs  <90   N= 22  rho=+0.3055  p=1.668e-01   
  Spirals ≥90   N= 43  rho=+0.0054  p=9.724e-01   

M3 BLIND: Partial rho(resid, Vflat | logMbar)
--------------------------------------------------
  M0: raw rho=+0.0643 p=6.109e-01  partial rho=+0.8930 p=1.615e-23
  M1: raw rho=+0.1485 p=2.379e-01  partial rho=+0.9318 p=2.029e-29
  M3: raw rho=+0.2526 p=4.233e-02  partial rho=+0.8828 p=2.387e-22

M1 BLIND: Remaining structure check
--------------------------------------------------
  rho(resid, Vflat       ): +0.1485  p=2.379e-01   
  rho(resid, Reff        ): -0.0448  p=7.233e-01   
  rho(resid, Mbar        ): -0.0901  p=4.753e-01   
  rho(resid, fgas        ): +0.1868  p=1.361e-01   
  rho(resid, T           ): +0.0483  p=7.023e-01   
  rho(resid, logMbar     ): -0.0901  p=4.753e-01   
  rho(resid, logReff     ): -0.0448  p=7.233e-01   
  rh